In [ ]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough

import argparse, inspect, json, os, pickle, socket, subprocess, warnings, random, math, librosa, shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import lightning as L
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers.tensorboard import TensorBoardLogger
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm

from functools import reduce
import umap
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

import commons, models, utils, losses, lightning_wrapper
from cough_datasets import CoughDatasets, CoughDatasetsCollate, CoughDiseaseBinaryBatchSampler

torch.set_float32_matmul_precision("medium")
cmap = cm.get_cmap("viridis")

In [ ]:
selected_experiment = ['resnet34_gtgram_deltadelta',
 'resnet34_gtgram_delta',
 'resnet34_gtgram',
 'bilstm_gtgram',
 'resnet34_mfcc']

selected_experiment = ['resnet34_gtgram_deltadelta']

In [ ]:
dfs = []
for now_experiment in selected_experiment:
    try:
        parser = argparse.ArgumentParser()
        parser.add_argument("--init", action="store_true")
        parser.add_argument("--model_name", type=str, default="try_wavlmlora_downstream")
        parser.add_argument("--config_path", type=str, default="configs/ssl_finetuning.json")
        args = parser.parse_args(["--model_name", now_experiment])

        model_dir = os.path.join("./logs", args.model_name)
        config_save_path = os.path.join(model_dir, "config.json")
        with open(config_save_path, "r") as f:
            data = f.read()
        config = json.loads(data)

        hps = utils.HParams(**config)
        hps.model_dir = model_dir
        hps.data.mae_training = hps.train.mae_training
        hps.data.ssccl_training = hps.train.ssccl_training
        hps.model.spk_dim = 0

        logger = utils.get_logger(hps.model_dir)
        import sys, importlib.util, shutil, tempfile
        temp_path = tempfile.NamedTemporaryFile(suffix=".py", delete=False).name
        shutil.copy(f"{model_dir}/model_net.py.bak", temp_path)
        spec = importlib.util.spec_from_file_location("model_net", temp_path)
        model_net = importlib.util.module_from_spec(spec)
        sys.modules["model_net"] = model_net
        spec.loader.exec_module(model_net)
        pool_net = getattr(model_net, hps.model.pooling_model)

        hps.model.lora_finetune = False
        pool_model = pool_net(hps.model.feature_dim, **hps.model)

        runner_lightning = lightning_wrapper.CoughClassificationRunner(pool_model, hps=hps, custom_logger=logger, class_weights=[])
        runner_lightning = lightning_wrapper.CoughClassificationRunner.load_from_checkpoint(
            os.path.join(f"{hps.model_dir}/best_model.ckpt"),
            model=pool_model,
            hps=hps, custom_logger=logger
        )
        runner_lightning.eval()

        df_train = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/{hps.data.metadata_csv}.train')
        df_test = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/{hps.data.metadata_csv}.test')

        df_train = df_train.reset_index(drop=True)
        df_test = df_test.reset_index(drop=True)

        df_train = df_train[hps.data.column_order]
        df_test = df_test[hps.data.column_order]

        collate_fn = CoughDatasetsCollate(hps.data.many_class)
        target_labels = df_train[hps.data.target_column]

        with open(os.path.join(hps.model_dir, "info_fold.pkl"), "rb") as f:
            info_fold = pickle.load(f)
            best_fold_idx = info_fold["best_fold_idx"]
            fold_metrics = info_fold["fold_metrics"]


        val_dataset = CoughDatasets(df_test.values, hps.data,
                                        wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{best_fold_idx}.pickle", train=False)
        val_loader = DataLoader(val_dataset, num_workers=28, shuffle=False,
                                batch_size=hps.train.batch_size, pin_memory=True, drop_last=True, collate_fn=collate_fn)
        
        test_wavnames = []
        test_preds = []
        with torch.no_grad():
            for idx, batch in tqdm(enumerate(val_loader), total=len(val_loader)):
                wavnames, audio, _, attention_masks, dse_ids, [patient_ids, _, _] = batch
                audio = audio.cuda()
                attention_masks = attention_masks.cuda()
                out_model = runner_lightning.model.forward(audio, attention_mask=attention_masks)
                logits = out_model['disease_logits']

                probs = torch.sigmoid(logits)     # [B]
                preds = (probs >= 0.5).long().squeeze(-1) 

                test_wavnames.extend(wavnames)
                test_preds.append(preds.cpu())

        del audio, attention_masks
        test_wavnames = np.array(test_wavnames)
        test_preds = torch.cat(test_preds, dim=0).numpy()

        df_now = pd.DataFrame({
            "path_file": test_wavnames,
            now_experiment: test_preds
        })
        dfs.append(df_now)
    except:
        pass
    
final_df = reduce(
    lambda left, right: pd.merge(left, right, on="path_file", how="inner"),
    dfs
)

In [ ]:
trainer = L.Trainer(
    max_epochs=1000,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices="auto",
    default_root_dir=hps.model_dir
)

In [ ]:
df_test = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/cirdz.csv.test')
df_test = df_test.reset_index(drop=True)
df_test = df_test[hps.data.column_order]
val_dataset = CoughDatasets(df_test.values, hps.data,
                            wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{best_fold_idx}.pickle", train=False)
val_loader = DataLoader(val_dataset, num_workers=28, shuffle=False, batch_size=hps.train.batch_size,
                        pin_memory=True, drop_last=True, collate_fn=collate_fn)
results = trainer.test(runner_lightning, dataloaders=val_loader)[0]

In [ ]:
exp_cols = [c for c in final_df.columns if c != "path_file"]
final_df["vote_sum"] = final_df[exp_cols].sum(axis=1)
final_df["vote_ratio"] = final_df["vote_sum"] / len(exp_cols)
final_df["ensemble_pred"] = (final_df["vote_ratio"] >= 0.8).astype(int)
final_df_ensem = final_df[['path_file', 'ensemble_pred']]

test_wavnames = []
test_labels = []
test_patient = []
with torch.no_grad():
    for idx, batch in tqdm(enumerate(val_loader), total=len(val_loader)):
        wavnames, audio, _, attention_masks, dse_ids, [patient_ids, _, _] = batch
        dse_ids = torch.argmax(dse_ids, dim=1)

        test_wavnames.extend(wavnames)
        test_labels.append(dse_ids.cpu())
        test_patient.append(patient_ids.cpu())

del audio, attention_masks
test_wavnames = np.array(test_wavnames)
test_labels = torch.cat(test_labels, dim=0).numpy()
test_patient = torch.cat(test_patient, dim=0).numpy()

meta_df = pd.DataFrame({
    "path_file": test_wavnames,
    "participant": test_patient,
    "disease_status": test_labels,
})

df = final_df_ensem.merge(
    meta_df[["path_file", "participant", "disease_status"]],
    on="path_file",
    how="inner",
    validate="one_to_one"
)

patient_vote = (
    df.groupby("participant")["ensemble_pred"]
      .mean()
      .ge(0.5)
      .astype(int)
      .rename("patient_ensemble_pred")
)

df = df.join(patient_vote, on="participant")
df["ensemble_pred"] = df["patient_ensemble_pred"]
df = df.drop(columns=["patient_ensemble_pred"])

df["final_label"] = df["ensemble_pred"]
df.loc[
    (df["disease_status"] == 1) & (df["ensemble_pred"] == 0),
    "final_label"
] = 2

df.loc[
    (df["disease_status"] == 0) & (df["ensemble_pred"] == 1),
    "final_label"
] = 4


In [ ]:
df[["path_file", "participant", "disease_status", "final_label"]].to_csv("consensus_label.csv", index=False)

In [ ]:
df['final_label'].value_counts()

In [ ]:
df_train = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/{hps.data.metadata_csv}.train')
df_train = df_train.reset_index(drop=True)

In [ ]:
df_train = df_train.merge(
    df_ensemble[['wavname', 'final_label']],
    left_on='path_file',
    right_on='wavname',
    how='left'
)

# 2. Replace disease_status with final_label when available
df_train['disease_status'] = (
    df_train['final_label']
    .combine_first(df_train['disease_status'])
)

# 3. Drop helper columns
df_train = df_train.drop(columns=['wavname', 'final_label'])


In [ ]:
df_train['disease_status'].value_counts()

In [ ]:
df_train = df_train[df_train['disease_status'] != 4].reset_index(drop=True)

In [ ]:
df_train.to_csv("/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/metadata.csv.train")